# Real Google traces: controlled resource placement

[Open in Colab](https://colab.research.google.com/github/crypticsymmetry/improved-giggle/blob/main/notebooks/google_cluster_benchmark.ipynb).

This CPU-only notebook downloads about 4.5 MB of public production trace files and compares whole-task placements on a bounded cohort. CPU/memory requests are **limits**, not measured consumption. Selected full machine capacities are modeled as hard limits; existing occupancy is not reconstructed. These are alternative plans, not a reproduction of Borg or measured production savings.

MILP, priority-first greedy, best-fit greedy and stability-aware best-fit receive identical inputs. The stability-aware heuristic prefers a feasible prior machine; the original heuristics do not. An independently verified optimal fractional LP gives a lower bound; fractional assignments are never rounded into executable placements. The initial MILP plan is a common frozen migration-cost reference.

In [ ]:
import importlib
import importlib.util
import subprocess
import sys

needs_install = importlib.util.find_spec("assumption_ops") is None
if not needs_install:
    import assumption_ops
    needs_install = not hasattr(assumption_ops, "run_cluster_benchmark")
if needs_install:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade",
        "git+https://github.com/crypticsymmetry/improved-giggle.git",
    ])
    for name in list(sys.modules):
        if name == "assumption_ops" or name.startswith("assumption_ops."):
            del sys.modules[name]
    importlib.invalidate_caches()

import json
import os
from pathlib import Path
from pprint import pprint
from IPython.display import Markdown, display
from assumption_ops.cluster_data import download_cluster, read_cluster_snapshot
from assumption_ops.cluster_benchmark import run_cluster_benchmark, write_cluster_csv

## Download pinned public files

The downloader verifies exact checksums and writes atomically under a size cap. It fetches the complete machine-events file and first task-events shard, not the full approximately 41 GB release. No account, GPU or API key is required. Set `ASSUMPTION_OPS_CLUSTER_DIR` to validate offline using the same pinned files.

In [ ]:
data_dir = Path(os.environ.get("ASSUMPTION_OPS_CLUSTER_DIR", "cluster_data"))
if os.environ.get("ASSUMPTION_OPS_CLUSTER_DIR"):
    paths = {
        "machines": data_dir / "google_machine_events.csv.gz",
        "tasks": data_dir / "google_task_events.csv.gz",
    }
else:
    paths = download_cluster(data_dir)
assert all(Path(path).is_file() for path in paths.values())
pprint({key: {"path": str(path), "bytes": Path(path).stat().st_size}
        for key, path in paths.items()})

## Inspect an as-of snapshot

Only source events through the cutoff are visible. The deterministic cohort contains newly submitted tasks with known positive CPU/memory requests; flagged different-machine constraints are excluded. The bounded cohort retains its first admitted identities; termination does not refill it. Selection details are recorded in metadata. Capacities round down and requests up at one million integer ticks. Quantities remain publisher-normalized values, not literal cores or bytes.

The release provides event timestamps but no per-record ingestion timestamps: actual observation latency is unknown.

In [ ]:
snapshot = read_cluster_snapshot(
    paths["machines"], paths["tasks"], cutoff_us=900_000_000,
    max_machines=4, max_tasks=64,
)
print(f"Selected machines: {len(snapshot.machines)}, tasks: {len(snapshot.tasks)}")
pprint(snapshot.metadata)
print("Example machines:", snapshot.machines[:2])
print("Example tasks:", snapshot.tasks[:3])

## Compare methods

Cutoffs of 900, 1,200 and 1,800 seconds are inside the first shard: 5, 10 and 20 minutes after the trace start at 600 seconds. Priority plus one is an assumed admission weight, not financial value. Migration weight is explicit. `feasible_limit` denotes a verified incumbent without an optimality claim. A missing LP bound stays missing.

In [ ]:
report = run_cluster_benchmark(
    paths["machines"], paths["tasks"],
    cutoffs_us=(900_000_000, 1_200_000_000, 1_800_000_000),
    max_machines=4, max_tasks=64, migration_penalty=1, time_limit=30,
)
assert report["real_source_records"] and report["controlled_model"]
assert not report["external_dispatch"]
pprint(report["environment"])
print("Source:", report["source"])
print("License:", report["license"])

In [ ]:
rows = ["| Cutoff (s) | Method | Status | Placed tasks | Placed priority | Score ↓ | LP bound | Gap to LP | Time (s) |",
        "|---:|---|---|---:|---:|---:|---:|---:|---:|"]
def shown(value):
    return "unavailable" if value is None else f"{value:.4g}"
for snap in report["snapshots"]:
    bound = snap["comparison"]["lp_relaxation"]["lower_bound"]
    for result in snap["comparison"]["results"]:
        metrics, costs = result["evaluation"]["metrics"], result["evaluation"]["costs"]
        rows.append(
            f"| {snap['cutoff_us'] / 1_000_000:g} | {result['method']} | {result['status']} | "
            f"{metrics['placed_tasks']} | {metrics['placed_priority']} | {costs['total']} | "
            f"{shown(bound)} | {shown(result['gap_to_lp_bound'])} | {result['elapsed_seconds']:.4f} |"
        )
display(Markdown("\n".join(rows)))

## Inspect evidence and interpretation

Each task's resource-feasibility support includes the machine-capacity fact and **all colocated task requests**. A changed neighbor invalidates the shared-capacity justification. Supports do not establish global optimality: every snapshot is reoptimized. Historical task termination is observed input, not a consequence of these proposals.

Snapshots overlap: placed tasks and priority are never summed as cumulative completed work.

In [ ]:
for snap in report["snapshots"]:
    print("Cutoff:", snap["cutoff_us"], "fingerprint:", snap["input_fingerprint"])
    print("Impact:", {
        key: snap["support_impact"][key]
        for key in ("invalidated_previous_decisions", "previous_assigned_decisions",
                    "changed_feasibility_fact_keys", "added_feasibility_fact_keys")
    })
    for result in snap["comparison"]["results"]:
        metrics = result["evaluation"]["metrics"]
        assert metrics["placed_tasks"] + metrics["pending_tasks"] == metrics["tasks"]
        assert metrics["used_cpu"] <= metrics["cpu_capacity"]
        assert metrics["used_memory"] <= metrics["memory_capacity"]
    if snap["placement_supports"]:
        pprint(snap["placement_supports"][0])
print("Final snapshot evaluations:")
pprint(report["summary"]["final_method_evaluations"])

## Export reproducible results

JSON retains placements, provenance, independent evaluations, bounds and support explanations. CSV has one row per method and cutoff: alternative decisions, not an additive execution ledger.

In [ ]:
output_dir = Path("cluster_results")
output_dir.mkdir(exist_ok=True)
json_path, csv_path = output_dir / "benchmark.json", output_dir / "benchmark.csv"
json_path.write_text(json.dumps(report, indent=2, allow_nan=False) + "\n", encoding="utf-8")
write_cluster_csv(report, csv_path)
print("Saved:", json_path.resolve(), csv_path.resolve())
if (importlib.util.find_spec("google") is not None
        and importlib.util.find_spec("google.colab") is not None):
    from google.colab import files
    files.download(str(json_path))
    files.download(str(csv_path))

## Limits

This establishes verified proposed placements and evidence handling on real public records. It omits existing occupancy, disk, networking, general affinity, durations and actual observation delays. Google can overcommit resources; this model cannot. Outcomes do not establish superiority to Google's scheduler, business ROI or cumulative completed work. Review the report's full limitations before interpreting scores.